# Global registration with RANSAC
We are going to use [open3d](http://www.open3d.org/) to handle point clouds and generation of point clouds
We are importing the packages and defining a function which helps us drawing the point clouds.

In [1]:
import copy

import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
# Helper for visualizing two point clouds together.
# Why copy.deepcopy(): open3d's transform()/paint_uniform_color() methods
# modify a point cloud in place. Deep-copying source/target first means calling
# this function never alters the original point clouds.
# Why the optional recolor flag: when comparing two point clouds, it can be
# hard to tell which points belong to 'source' vs. 'target' if they're both
# rendered in their original scan colors. Painting them two distinct, contrasting
# uniform colors (orange and blue) makes the alignment between the two clouds
# much easier to judge by eye.
# Why the optional transformation: passing a 4x4 transformation matrix applies
# that transform to a copy of 'source' before displaying it. This is how we
# visually check whether a computed registration actually aligns the two point
# clouds correctly.
def draw_registrations(source, target, transformation=None, recolor=False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if recolor:
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if transformation is not None:
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

conda install -c conda-forge --force-reinstall scipy scikit-learn numpy

We need to read in our pointclouds. For that we use the `io` module of the
open3d package (`o3d`). The following cell will open a window with a
visualization of both point clouds `source` and `target`.

Also, this page [Visualization - Open3D](http://open3d.org/html/tutorial/Basic/visualization.html)
contains some useful examples and instructions on how to use the viewer.

In [3]:
# Load the two point clouds we'll try to align: 'source' (r1) and 'target' (r2).
# Why point clouds instead of images: these files represent 3D scans (a set of
# (x, y, z) points, typically from a depth sensor/LiDAR) rather than 2D pixel
# grids. The goal is to find the rigid transformation (rotation + translation)
# that best aligns two such scans of the same physical scene taken from
# different viewpoints.
source = o3d.io.read_point_cloud("ICP/r1.pcd")
target = o3d.io.read_point_cloud("ICP/r2.pcd")

# Uncomment it if you want to see the two  (still misaligned) point clouds side
# by side before any registration is attempted.
# draw_registrations(source, target, recolor=True)

### Finding features in pointclouds
When working on point clouds it can be beneficial to work on a downsampled version of the point cloud,
as it decreases the need of computation.

You can use [`pointcloud.voxel_down_sample()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample) where `pointcloud` is the name of your point cloud object. In our case, that would be `source` and `target`.

We also need to estimate the normals of the point cloud points using [`pointcloud.estimate_normals()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample)

**Task:** Find FPFH features or correspondances of the downsampled point clouds.
[`o3d.pipelines.registration.compute_fpfh_feature()`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.compute_fpfh_feature.html)


In [4]:
# Define a preprocessing pipeline: downsample each point cloud and extract FPFH
# features for use in RANSAC-based matching.
# Why downsample first (voxel_down_sample): raw point clouds can contain millions
# of points. Working with every single one would make normal estimation, feature
# computation, and matching extremely slow. Voxel downsampling divides 3D space
# into a grid of small cubes ("voxels") and keeps roughly one representative
# point per voxel, giving a much smaller but still representative point cloud.
# Why estimate normals (estimate_normals): a normal is the direction a small
# local surface patch is "facing". Many downstream operations need this local
# surface orientation information, which isn't directly present in a raw
# (x, y, z) point cloud and has to be estimated from each point's neighborhood.
# Why these specific parameters scale with voxel_size: radius_normal (2x voxel_size)
# and radius_feature (5x voxel_size) are both defined relative to the
# downsampling voxel size rather than as fixed constants. This keeps the
# neighborhood search radius proportional to the point spacing after
# downsampling, so the same preprocessing function behaves sensibly across point
# clouds of different original density/scale.
# Why FPFH features (compute_fpfh_feature): Fast Point Feature Histograms
# summarize the local geometric shape around each point (based on relationships
# between it and its neighbors' normals) into a numeric descriptor. Conceptually
# the 3D-point-cloud equivalent of a 2D feature descriptor like SIFT, letting us
# later find candidate correspondences between the two point clouds based on
# how similar their local shapes are, without needing any initial alignment guess.
def preprocess_point_cloud(pcd, voxel_size):
    print(":: Downsample with a voxel size %.3f." % voxel_size)
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
    )

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(
            radius=radius_feature, max_nn=100
        ),
    )
    return pcd_down, pcd_fpfh

In [5]:
# Run the preprocessing pipeline on both point clouds.
# Why voxel_size = 0.05: this sets the target spacing between points after
# downsampling (in the point cloud's native units, typically meters). It's the
# single parameter that both radius_normal and radius_feature above are scaled
# from, so choosing it appropriately is important: too large and fine geometric
# detail is lost; too small and processing stays slow.
voxel_size = 0.05
source_down, source_fpfh = preprocess_point_cloud(source, voxel_size)
target_down, target_fpfh = preprocess_point_cloud(target, voxel_size)

:: Downsample with a voxel size 0.050.
:: Estimate normal with search radius 0.100.
:: Compute FPFH feature with search radius 0.250.
:: Downsample with a voxel size 0.050.
:: Estimate normal with search radius 0.100.
:: Compute FPFH feature with search radius 0.250.


### RANSAC 
We will now attempt to use RANSAC to do a global registration of the two point clouds.

By using the function [`o3d.pipelines.registration.registration_ransac_based_on_feature_matching`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.registration_ransac_based_on_feature_matching.html) from open3d, do the following:


Try to find the transformation from `r1.pcd` (`source`) to `r2.pcd` (`target`).
Attempt with point-to-point and point-to-plane
```Python
point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()
```

When using RANSAC, focus on the arguments below. The rest are optional parameters.
```Python
ransac_result = o3d.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    distance_threshold,
    point_to_point)
```

In [6]:
# Why "global" registration: unlike local refinement methods (e.g. ICP), which
# need a reasonably good initial alignment guess to converge correctly, RANSAC-based
# feature matching can find a good alignment from a completely arbitrary starting
# position/orientation, by matching points based on their FPFH feature similarity
# rather than their raw spatial proximity.
# How RANSAC handles this: it repeatedly samples a small random set of candidate
# point correspondences (based on feature similarity), computes the rigid
# transformation that would align them, and checks how many other points from
# the two clouds also end up close together under that transformation ("inliers").
# The transformation with the most inlier support is kept as the result, which
# makes it robust to the many incorrect/spurious feature matches that occur.
# Why point_to_point vs. point_to_plane: TransformationEstimationPoint-
# ToPoint minimizes the raw Euclidean distance between corresponding points;
# TransformationEstimationPointToPlane instead minimizes the distance along each
# point's surface normal direction, which can converge faster and more accurately
# on smooth surfaces since it accounts for local surface orientation rather than
# treating points as unstructured dots. It requires reliable normal estimates to
# work well.
# Why distance_threshold = voxel_size * 1.5: this defines how close two points
# need to be (after applying a candidate transformation) to be counted as a
# RANSAC inlier. Scaling it relative to voxel_size keeps it consistent with the
# downsampling resolution used above, since points closer together than the
# downsampling voxel size would be considered noise/duplicates.
distance_threshold = voxel_size * 1.5

point_to_point = (
    o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
)
point_to_plane = (
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

result = (
    o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down,
        target_down,
        source_fpfh,
        target_fpfh,
        True,
        distance_threshold,
        point_to_point
    )
)

print(result)
# draw_registrations(source, target, result.transformation,True)

RegistrationResult with fitness=6.747899e-01, inlier_rmse=2.963504e-02, and correspondence_set size of 3212
Access transformation to get result.


## Exercises
### A)
Can you get a decent transformation from r1 to r3? (check the ICP folder)

In [9]:
# Why reprocess target2 separately: r3 is a different point cloud file from r2,
# so it needs to go through the exact same preprocessing pipeline  as source/target
# did above, before RANSAC-based matching against it is possible. Note 'source'
# and its FPFH features are reused as-is from the earlier cells, since we're
# matching the same source scan against a new target. The remaining registration
# call is otherwise identical in structure to the r1-to-r2 registration above.
target2 = o3d.io.read_point_cloud("ICP/r3.pcd")

voxel_size = 0.05
distance_threshold = voxel_size * 1.5
target2_down, target2_fpfh = preprocess_point_cloud(target2, voxel_size)

result = (
    o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down,
        target2_down,
        source_fpfh,
        target2_fpfh,
        True,
        distance_threshold,
        point_to_point
    )
)

print(result)
print(":: Transformation between r1 and r3 is:")
print(result.transformation)

# draw_registrations(source, target2, result.transformation,True)

:: Downsample with a voxel size 0.050.
:: Estimate normal with search radius 0.100.
:: Compute FPFH feature with search radius 0.250.
RegistrationResult with fitness=7.636555e-01, inlier_rmse=3.491782e-02, and correspondence_set size of 3635
Access transformation to get result.
:: Transformation between r1 and r3 is:
[[-0.30430055  0.95238547  0.01905525 -0.01924691]
 [-0.23412134 -0.09416545  0.96763633  0.85737797]
 [ 0.92335712  0.28999102  0.25162836 -1.20497789]
 [ 0.          0.          0.          1.        ]]



### B)
With the following checkers, can you get better results from RANSAC? Try tweaking the parameters of them. Can you make point-to-plane work? Do not spend too much time on this, if you can't manage, skip it. (I was not able to get a good fit.)

You can also try tweaking the `voxel_size`

```Python
corr_length = 0.9
distance_threshold = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]

ransac_result = o3d.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)
```


In [8]:
# Why correspondence checkers: RANSAC only checks how many points support a
# hypothesized transformation, but doesn't otherwise validate whether an
# individual candidate correspondence is geometrically plausible on its own.
# Checkers add fast pre-filtering rules that reject bad candidate matches early,
# which can make RANSAC converge to a better/more accurate transformation.
#   CorrespondenceCheckerBasedOnEdgeLength(corr_length): for a sampled set of
#     point correspondences, compares the pairwise distances between points
#     within the source sample against the corresponding distances within the
#     target sample. Rejects the sample if these "edge lengths" don't roughly
#     match, since a true rigid transformation must preserve distances between
#     points.
#   CorrespondenceCheckerBasedOnDistance(distance_threshold): rejects a
#     correspondence if, after applying the candidate transformation, the matched
#     points end up farther apart than distance_threshold. Essentially a stricter,
#     per-correspondence version of the inlier check RANSAC does globally.
#   CorrespondenceCheckerBasedOnNormal(0.095): rejects a correspondence if the
#     local surface normals of the two matched points don't point in sufficiently
#     similar directions (within the given angular threshold). Since real
#     matching surface points should have similar local orientation, this helps
#     reject matches on geometrically implausible/mismatched surface patches.
# Why ransac_n = 3: this sets how many point correspondences are sampled
# together to hypothesize each candidate transformation. 3 is the minimum needed
# to fully determine a rigid transformation in 3D, keeping each RANSAC iteration
# cheap, though it also makes each individual hypothesis more sensitive to any
# single bad correspondence slipping through the checkers.
voxel_size = 0.05
corr_length = 0.9
distance_threshold = voxel_size * 1.5

print(":: RANSAC registration on downsampled point clouds.")
print("   Since the downsampling voxel size is %.3f," % voxel_size)
print("   we use a liberal distance threshold %.3f." % distance_threshold)

corr_check = [
    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
        corr_length
    ),
    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
        distance_threshold
    ),
    o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095),
]


result = (
    o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down,
        target_down,
        source_fpfh,
        target_fpfh,
        True,  # mutual_filter
        distance_threshold,
        point_to_point,
        3,  # ransac_n
        corr_check,
    )
)

print(result)
# draw_registrations(source, target, result.transformation, True)

:: RANSAC registration on downsampled point clouds.
   Since the downsampling voxel size is 0.050,
   we use a liberal distance threshold 0.075.
RegistrationResult with fitness=6.735294e-01, inlier_rmse=2.851020e-02, and correspondence_set size of 3206
Access transformation to get result.
